# 02 — Evaluation: mAP, PR curves, and per-class AP

Walk through validation of the trained YOLOv8 detector:

1. Load the best checkpoint
2. Run `model.val()` on the VOC validation split
3. Report precision, recall, mAP@0.5, and mAP@0.5:0.95
4. Inspect per-class AP, PR/F1 curves, the confusion matrix, and loss curves

This notebook is self-contained: it does not import from `src/`.


In [ ]:
%pip install -q ultralytics pandas matplotlib


## 1. Setup

Expects the dataset from `notebooks/01-data-and-training.ipynb` (`data/dataset.yaml`)
and a trained checkpoint in `models/`.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'models').exists() and (ROOT.parent / 'models').exists():
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / 'results'
MODELS_DIR = ROOT / 'models'
DATASET_YAML = ROOT / 'data' / 'dataset.yaml'
CHECKPOINT = MODELS_DIR / 'yolov8n_voc_best.pt'
CONF = 0.25
IOU = 0.45

print('Project root:', ROOT)
print('Checkpoint:', CHECKPOINT)


## 2. Load the best checkpoint

Falls back to `yolov8n.pt` so the notebook still runs as a smoke test.


In [ ]:
from ultralytics import YOLO

weights = CHECKPOINT if CHECKPOINT.exists() else MODELS_DIR / 'yolov8n.pt'
if not weights.exists():
    raise FileNotFoundError('No checkpoint found. Train first with notebook 01 or src/train.py.')

model = YOLO(str(weights))
names = [model.names[i] for i in sorted(model.names)] if isinstance(model.names, dict) else list(model.names)
print('Weights:', weights)
print('Classes:', len(names))


## 3. Validate on Pascal VOC

`model.val()` computes precision, recall, mAP@0.5, and mAP@0.5:0.95 and writes
curves and a confusion matrix to the run directory.


In [ ]:
if not DATASET_YAML.exists():
    raise FileNotFoundError('Run notebook 01 or python src/dataset.py first to create data/dataset.yaml.')

metrics = model.val(
    data=str(DATASET_YAML),
    imgsz=640,
    batch=16,
    conf=CONF,
    iou=IOU,
    project=str(RESULTS_DIR / 'runs'),
    name='eval_quick',
    plots=True,
)
run_dir = Path(getattr(metrics, 'save_dir', RESULTS_DIR / 'runs' / 'eval_quick'))
print('Run directory:', run_dir)


## 4. Headline metrics


In [ ]:
summary = {
    'precision': metrics.results_dict.get('metrics/precision(B)'),
    'recall': metrics.results_dict.get('metrics/recall(B)'),
    'mAP50': metrics.results_dict.get('metrics/mAP50(B)'),
    'mAP50-95': metrics.results_dict.get('metrics/mAP50-95(B)'),
}
pd.Series(summary, name='value').to_frame()


## 5. Per-class AP

`maps` is per-class AP across IoU thresholds 0.5:0.95.


In [ ]:
per_class = pd.DataFrame({
    'class': names,
    'mAP50-95': metrics.box.maps,
}).sort_values('mAP50-95', ascending=False)
per_class.reset_index(drop=True)


## 6. PR, F1, and confusion-matrix artifacts


In [ ]:
for filename in ('BoxPR_curve.png', 'BoxF1_curve.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png'):
    path = run_dir / filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))
    else:
        print('Missing:', path)


## 7. Loss curves

Parse `results.csv` and plot box, classification, and DFL losses.


In [ ]:
loss_groups = {
    'box loss': ('train/box_loss', 'val/box_loss'),
    'cls loss': ('train/cls_loss', 'val/cls_loss'),
    'dfl loss': ('train/dfl_loss', 'val/dfl_loss'),
}
csv_path = run_dir / 'results.csv'
if csv_path.exists():
    history = pd.read_csv(csv_path)
    history.columns = [column.strip() for column in history.columns]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, (group, columns) in zip(axes, loss_groups.items()):
        for column in columns:
            if column in history.columns:
                ax.plot(history['epoch'], history[column], label=column.split('/', 1)[0])
        ax.set_title(group.title())
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(RESULTS_DIR / 'loss_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Missing:', csv_path)


## 8. Write results/metrics.json


In [ ]:
payload = {
    'run': 'eval_quick',
    'weights': str(weights),
    'dataset': str(DATASET_YAML),
    'metrics': summary,
    'per_class_ap50_95': dict(zip(names, metrics.box.maps.tolist())),
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / 'metrics.json'
out_path.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
print('Saved:', out_path)


## Next steps

- Inspect false positives and false negatives in `03-error-analysis.ipynb` (Day 2).
- Tune confidence and IoU thresholds based on the PR curve.
- Run `python src/evaluate.py --weights models/yolov8n_voc_best.pt` for the CLI path.
- Track the full training run in `results/metrics.json` and the README results table.
